In [ ]:

!pip -q install transformers accelerate pandas numpy scikit-learn matplotlib tqdm

# 2. Imports
import os
import re
import math
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import calinski_harabasz_score

# 3. Configuration
FILE_IN  = "/content/drive/MyDrive/embadding reltion/testing-sam-00000200.xlsx"
HF_TOKEN = ""  #
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE   = 0 if torch.cuda.is_available() else -1

# Output paths
FILE_OUT = FILE_IN.replace(".xlsx", "_with_preds.xlsx")
CSV_CH   = FILE_IN.replace(".xlsx", "_CH_values.csv")
PNG_TSNE = FILE_IN.replace(".xlsx", "_tSNE_grid.png")

# 4. Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
# Ensure padding token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = (
    AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN)
    .half()
).to("cuda" if DEVICE == 0 else "cpu")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=DEVICE)

# 5. Prompt template
PROMPT = (
    "F3eI?%qt,NbnG8U"
    "Gene Interaction Scientist, Your role is to explore how {geneA} modulates "
    "the function of {geneB} using data from the KEGG Pathway Database, focusing "
    "on molecular interactions. Clarify the type of interaction between {geneA} "
    "and {geneB} by selecting from \"activation\", \"inhibition\", or "
    "\"phosphorylation\". Provide only that term.\n"
    "Q: What relationship exists between gene {geneA} and gene {geneB}?"
)

def ask_llama(row):
    prompt = PROMPT.format(geneA=row["Gene-A"], geneB=row["Gene-B"])
    out    = pipe(prompt, max_new_tokens=4, do_sample=False)[0]["generated_text"]
    match  = re.search(r"(activation|inhibition|phosphorylation)", out, re.I)
    return match.group(1).capitalize() if match else "No information"

# 6. Read Excel and predict
df = pd.read_excel(FILE_IN)
tqdm.pandas(desc="Llama‑3 predictions")
df["Prediction"] = df.progress_apply(ask_llama, axis=1)
df["Label"] = np.where(
    df["Prediction"] == df["Ground truth"], "Correct", "Incorrect"
)

# 7. Build text for embeddings
df["Text"] = (
    df["Gene-A"] + " interacts with " + df["Gene-B"] +
    ", ground truth is " + df["Ground truth"] +
    ", prediction is " + df["Prediction"]
)

# 8. Layer‑wise embeddings

def embed(texts):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    with torch.no_grad():
        hidden_states = model(**enc, output_hidden_states=True).hidden_states
    return [h.mean(dim=1).cpu().numpy() for h in hidden_states]

all_layers = [embed([text]) for text in tqdm(df["Text"], desc="Embedding rows")]
NUM_LAYERS = len(all_layers[0])
NUM_ROWS   = len(df)
EMB_DIM    = all_layers[0][0].shape[1]
emb = np.zeros((NUM_LAYERS, NUM_ROWS, EMB_DIM))
for r, layers in enumerate(all_layers):
    for l, vec in enumerate(layers):
        emb[l, r] = vec

# 9. Compute Calinski–Harabasz index per layer 1–33
gt_codes = df["Ground truth"].astype("category").cat.codes
ch_vals  = []
for layer in range(1, NUM_LAYERS):
    X_scaled = StandardScaler().fit_transform(emb[layer])
    ch_vals.append(calinski_harabasz_score(X_scaled, gt_codes))

ch_df = pd.DataFrame({"Layer": range(1, NUM_LAYERS), "CH": ch_vals})
ch_df.to_csv(CSV_CH, index=False)
print(f"✔ CH values saved to {CSV_CH}")

# 10. 5‑column t‑SNE grid plot
GRID_COLS = 5
GRID_ROWS = math.ceil((NUM_LAYERS - 1) / GRID_COLS)
plt.figure(figsize=(GRID_COLS * 8, GRID_ROWS * 7))

markers = {'Activation':'o','Inhibition':'s','Phosphorylation':'^','Incorrect':'x'}
colors  = {'Activation':'blue','Inhibition':'red','Phosphorylation':'black','Incorrect':'purple'}

for idx, layer in enumerate(range(1, NUM_LAYERS)):
    Xs = StandardScaler().fit_transform(emb[layer])
    pts = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate=200,
        n_iter=1000,
        random_state=42
    ).fit_transform(Xs)

    plt.subplot(GRID_ROWS, GRID_COLS, idx + 1)
    for rel in ['Activation','Inhibition','Phosphorylation']:
        subset = df[(df['Ground truth'] == rel) & (df['Label'] == 'Correct')]
        plt.scatter(
            pts[subset.index, 0], pts[subset.index, 1],
            marker=markers[rel],
            c=colors[rel],
            alpha=0.8,
            label=rel if idx == 0 else None
        )
    wrong = df[df['Label'] == 'Incorrect']
    plt.scatter(
        pts[wrong.index, 0], pts[wrong.index, 1],
        marker='x',
        c=colors['Incorrect'],
        alpha=0.8,
        label='Incorrect' if idx == 0 else None
    )
    plt.title(f"Layer {layer} | CH={ch_vals[layer-1]:.2f}", fontsize=10)
    plt.xticks([]); plt.yticks([])

# One legend for all panels
handles, labels = plt.gca().get_legend_handles_labels()
plt.figlegend(handles, labels, loc='lower center', ncol=4, fontsize=11)
plt.subplots_adjust(bottom=0.05)

plt.tight_layout()
plt.savefig(PNG_TSNE, dpi=600)
plt.show()
print(f"✔ t‑SNE grid saved to {PNG_TSNE}")

# 11. Save workbook with predictions
df.to_excel(FILE_OUT, index=False)
print(f"✔ Workbook with predictions saved to {FILE_OUT}")